# Reproduce Results

This notebook **loads pre-trained models from disk** and evaluates them on the
train / validation / test splits.  No training happens here.

Metrics reported:
- **ROC-AUC** (`sklearn.metrics.roc_auc_score`)
- **Precision** (`sklearn.metrics.precision_score`)
- **Recall** (`sklearn.metrics.recall_score`)
- **F1** (`sklearn.metrics.f1_score`)

In [ ]:
import os
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split

import utils
from utils import (
    load_object,
    get_combined_features,
    evaluate_model,
)

## 1. Data loading and splitting

The exact same random seed and split ratios as `train_models.ipynb` are used
so that train / val / test sets are identical across notebooks.

In [ ]:
DATA_PATH = os.path.expanduser("~/Datasets/QuoraQuestionPairs/quora_data.csv")
quora_df = pd.read_csv(DATA_PATH)

A_df, test_df = train_test_split(quora_df, test_size=0.05, random_state=123)
train_df, val_df = train_test_split(A_df,  test_size=0.05, random_state=123)

print(f'train_df.shape = {train_df.shape}')
print(f'val_df.shape   = {val_df.shape}')
print(f'test_df.shape  = {test_df.shape}')

y_train = train_df["is_duplicate"].values
y_val   = val_df["is_duplicate"].values
y_test  = test_df["is_duplicate"].values

## 2. Load models and vectorizers

In [ ]:
MODELS_DIR = "models"

count_vectorizer    = load_object(os.path.join(MODELS_DIR, "count_vectorizer.pkl"))
tfidf_vectorizer    = load_object(os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl"))
baseline_logistic   = load_object(os.path.join(MODELS_DIR, "baseline_logistic.pkl"))
improved_logistic   = load_object(os.path.join(MODELS_DIR, "improved_logistic.pkl"))

print("All models and vectorizers loaded successfully.")

In [ ]:

# Some examples of mistakes that the model makes on validation
mistake_indices, predictions = utils.get_mistakes(logistic, X_val, y_val)

utils.print_mistake_k(0, val_df, mistake_indices, predictions)
utils.print_mistake_k(9, val_df, mistake_indices, predictions)
utils.print_mistake_k(13, val_df, mistake_indices, predictions)
utils.print_mistake_k(19, val_df, mistake_indices, predictions)
utils.print_mistake_k(25, val_df, mistake_indices, predictions)


## 3. Feature extraction

- **Baseline**: sparse BoW matrix (CountVectorizer, unigrams)
- **Improved**: BoW + 5 handcrafted similarity features (see `utils.py`)

In [ ]:
print("Extracting baseline (BoW) features...")
X_train_bow = utils.get_features_from_df(train_df, count_vectorizer)
X_val_bow   = utils.get_features_from_df(val_df,   count_vectorizer)
X_test_bow  = utils.get_features_from_df(test_df,  count_vectorizer)
print(f"  train shape: {X_train_bow.shape}")

print("\nExtracting combined (BoW + handcrafted) features...")
X_train_comb = get_combined_features(train_df, count_vectorizer, tfidf_vectorizer)
X_val_comb   = get_combined_features(val_df,   count_vectorizer, tfidf_vectorizer)
X_test_comb  = get_combined_features(test_df,  count_vectorizer, tfidf_vectorizer)
print(f"  train shape: {X_train_comb.shape}")

## 4. Evaluation — ROC-AUC, Precision, Recall, F1

In [ ]:
rows = []

# Baseline model on all three splits
rows.append(evaluate_model(baseline_logistic, X_train_bow, y_train, "baseline", "train"))
rows.append(evaluate_model(baseline_logistic, X_val_bow,   y_val,   "baseline", "val"))
rows.append(evaluate_model(baseline_logistic, X_test_bow,  y_test,  "baseline", "test"))

# Improved model on all three splits
rows.append(evaluate_model(improved_logistic, X_train_comb, y_train, "improved", "train"))
rows.append(evaluate_model(improved_logistic, X_val_comb,   y_val,   "improved", "val"))
rows.append(evaluate_model(improved_logistic, X_test_comb,  y_test,  "improved", "test"))

results_df = pd.DataFrame(rows)
results_df = results_df.set_index(["model", "split"])
display(results_df)

## 5. Summary pivot

In [ ]:
pivot = results_df.reset_index().pivot(index="split", columns="model")
# Reorder rows: train → val → test
pivot = pivot.loc[["train", "val", "test"]]
display(pivot)